# PRAGMA Phase 1 - Synthetic data generation

Generates a synthetic raw-event and profile-state corpus shaped like section 5 of
`plans/PRAGMA-Implementation-Plan.md`: heterogeneous banking events (top-ups, card
payments, P2P transfers, ATM withdrawals, FX exchanges, direct debits, app events)
plus point-in-time profile state with lifelong milestones.

The generator (`pragma.data.synthetic`) deliberately covers the edge cases later
phases must handle: zero-event entities, single-event entities, very long histories,
colliding timestamps, missing optional fields, and rare/OOV categorical values.

This notebook is a thin, inspectable wrapper: all generation and validation logic
lives in `src/pragma/`, so it is unit-tested independently of this notebook.

In [1]:
import json
from pathlib import Path

import pandas as pd

from pragma.data.synthetic import SyntheticDataConfig, generate_synthetic_corpus, validate_corpus

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUT_DIR = REPO_ROOT / "data" / "raw"
OUT_DIR

WindowsPath('C:/Users/levyr/Desktop/random-projects/pragma/data/raw')

## Generate the corpus

In [2]:
config = SyntheticDataConfig(n_entities=500, seed=1337)
events_df, profile_df, manifest = generate_synthetic_corpus(config)

print(json.dumps(manifest, indent=2, default=str))

{
  "seed": 1337,
  "n_entities": 500,
  "n_events": 108022,
  "history_start": "2024-01-01T00:00:00+00:00",
  "history_end": "2026-06-30T23:59:59+00:00",
  "edge_case_counts": {
    "normal": 455,
    "single_event": 15,
    "zero_events": 15,
    "same_timestamp": 10,
    "long_history": 5
  },
  "event_family_counts": {
    "card_payment": 48664,
    "app_event": 21420,
    "topup": 16330,
    "p2p_transfer": 10692,
    "atm_withdrawal": 5546,
    "direct_debit": 3244,
    "fx_exchange": 2126
  },
  "schema_version": 1
}


In [3]:
events_df.head(10)

,entity_id,event_id,created_at,type,amount,currency,direction,counterparty_id,description,view,mcc,merchant_name,channel,source_currency
0,u000000,e00000144,2024-02-28 23:18:16+00:00,topup,35.68,GBP,in,NaN,dinner with friends,NaN,NaN,NaN,open_banking,NaN
1,u000000,e00000127,2024-02-29 19:13:03+00:00,card_payment,3.44,EUR,out,NaN,metal plan,NaN,5411,Tesco,NaN,NaN
2,u000000,e00000132,2024-03-03 23:18:10+00:00,card_payment,23.45,PLN,out,NaN,dinner with friends,NaN,5812,Amazon,NaN,NaN
3,u000000,e00000197,2024-03-04 23:46:33+00:00,topup,10.70,USD,in,NaN,dinner with friends,NaN,NaN,NaN,bank_transfer,NaN
4,u000000,e00000215,2024-03-10 00:31:09+00:00,card_payment,114.91,RON,out,NaN,monthly subscription,NaN,6012,Uber,NaN,NaN
5,u000000,e00000139,2024-03-12 08:18:53+00:00,card_payment,220.75,GBP,out,NaN,NaN,NaN,5411,Netflix,NaN,NaN
6,u000000,e00000158,2024-03-15 02:52:30+00:00,direct_debit,7.20,RON,out,NaN,grocery shopping,NaN,NaN,Tesco,NaN,NaN
7,u000000,e00000207,2024-03-15 22:40:11+00:00,app_event,NaN,NaN,NaN,NaN,NaN,support,NaN,NaN,NaN,NaN
8,u000000,e00000182,2024-03-26 01:05:24+00:00,card_payment,36.92,RON,out,NaN,rent payment,NaN,4111,Netflix,NaN,NaN
9,u000000,e00000076,2024-04-03 07:08:41+00:00,app_event,NaN,NaN,NaN,NaN,NaN,crypto,NaN,NaN,NaN,NaN


In [4]:
profile_df.head(10)

,entity_id,signup_at,country,plan,kyc_level,age_band,balance_quantile,is_active,first_topup_at,first_card_payment_at,first_p2p_at,_edge_case_profile
0,u000000,2024-02-23 15:33:54+00:00,GB,premium,basic,55+,q2,True,2024-02-28 23:18:16+00:00,2024-02-29 19:13:03+00:00,2024-05-01 01:51:05+00:00,normal
1,u000001,2024-06-27 16:26:14+00:00,RO,plus,enhanced,25-34,q2,True,2024-07-31 07:09:53+00:00,2024-06-27 18:20:19+00:00,2024-07-16 07:57:54+00:00,normal
2,u000002,2024-06-18 18:29:36+00:00,RO,premium,full,18-24,q3,True,2024-07-15 02:29:31+00:00,2024-06-20 02:20:48+00:00,2024-06-25 00:32:46+00:00,normal
3,u000003,2025-08-21 14:24:40+00:00,DE,standard,full,55+,q3,True,2025-08-23 08:38:21+00:00,2025-08-25 01:25:03+00:00,2026-01-10 16:53:42+00:00,normal
4,u000004,2025-01-05 16:24:44+00:00,RO,standard,full,45-54,q3,True,2025-01-06 23:02:18+00:00,2025-01-06 07:31:21+00:00,2025-01-19 06:46:23+00:00,normal
5,u000005,2025-10-15 12:38:08+00:00,RO,metal,basic,45-54,q2,True,2025-12-25 09:56:38+00:00,2025-10-16 16:19:26+00:00,2025-10-15 16:08:22+00:00,normal
6,u000006,2026-06-21 07:39:03+00:00,IE,premium,full,25-34,q2,True,2026-06-21 11:16:27+00:00,2026-06-21 09:56:37+00:00,2026-06-22 01:03:35+00:00,normal
7,u000007,2024-04-18 09:22:45+00:00,IE,plus,enhanced,35-44,q1,True,2024-04-28 07:22:39+00:00,2024-05-06 03:27:32+00:00,2024-04-20 12:39:26+00:00,normal
8,u000008,2024-07-16 10:28:22+00:00,IE,premium,full,18-24,q4,True,2024-07-30 08:01:09+00:00,2024-07-23 05:06:06+00:00,2024-07-20 03:44:00+00:00,normal
9,u000009,2024-05-29 05:23:34+00:00,RO,plus,full,25-34,q5,True,2024-06-03 11:50:38+00:00,2024-05-31 15:56:01+00:00,2024-06-03 09:38:12+00:00,normal


## Sanity checks

Event-type mix, history-length distribution (including the zero/single/long-history
edge-case cohorts), and null-rate spot checks per field.

In [5]:
events_df["type"].value_counts(normalize=True).rename("share").to_frame()

,share
type,
card_payment,0.450501
app_event,0.198293
topup,0.151173
p2p_transfer,0.098980
atm_withdrawal,0.051341
direct_debit,0.030031
fx_exchange,0.019681


In [6]:
history_lengths = events_df.groupby("entity_id").size()
history_lengths = history_lengths.reindex(profile_df["entity_id"], fill_value=0)

summary = profile_df[["entity_id", "_edge_case_profile"]].copy()
summary["n_events"] = summary["entity_id"].map(history_lengths)
summary.groupby("_edge_case_profile")["n_events"].describe()

,count,mean,std,min,25%,50%,75%,max
_edge_case_profile,,,,,,,,
long_history,5.0,3000.000000,0.000000,3000.0,3000.0,3000.0,3000.00,3000.0
normal,455.0,199.428571,115.209323,2.0,96.5,197.0,295.50,399.0
same_timestamp,10.0,226.700000,129.101209,96.0,131.5,147.0,366.25,398.0
single_event,15.0,1.000000,0.000000,1.0,1.0,1.0,1.00,1.0
zero_events,15.0,0.000000,0.000000,0.0,0.0,0.0,0.00,0.0


## Schema and leakage validation

Runs the same `SchemaRegistry`-backed checks used by `scripts/generate_synthetic_data.py`:
every raw field resolves to a canonical key and conforms to its event family, no event
precedes its entity's signup, and lifelong milestones match the earliest matching event.
The corpus must pass before it is written to disk.

In [7]:
report = validate_corpus(events_df, profile_df)
print(json.dumps(report, indent=2, default=str))

assert report["passed"], "Synthetic corpus failed schema/leakage validation - see report above."

{
  "n_schema_issues": 0,
  "schema_issues_sample": [],
  "n_leakage_issues": 0,
  "leakage_issues": [],
  "null_rate_by_field": {
    "amount": 0.19829294032697042,
    "currency": 0.19829294032697042,
    "direction": 0.19829294032697042,
    "counterparty_id": 0.9010201625594786,
    "description": 0.37464590546370186,
    "view": 0.8017070596730296,
    "mcc": 0.5494991760937586,
    "merchant_name": 0.5194682564662754,
    "channel": 0.7974856973579456,
    "source_currency": 0.9803188239432709
  },
  "entities_with_duplicate_event_timestamps": 14,
  "n_events": 108022,
  "n_entities": 500,
  "passed": true
}


## Write to `data/raw/`

In [8]:
OUT_DIR.mkdir(parents=True, exist_ok=True)

events_df.to_parquet(OUT_DIR / "events.parquet", index=False)
profile_df.to_parquet(OUT_DIR / "profile_state.parquet", index=False)
(OUT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2, default=str))
(OUT_DIR / "validation_report.json").write_text(json.dumps(report, indent=2, default=str))

print(f"Wrote {manifest['n_events']} events and {manifest['n_entities']} entities to {OUT_DIR}")

Wrote 108022 events and 500 entities to C:\Users\levyr\Desktop\random-projects\pragma\data\raw


In [9]:
import pandas as pd

pd.read_parquet("C:/Users/levyr/Desktop/random-projects/pragma/data/raw/profile_state.parquet")

,entity_id,signup_at,country,plan,kyc_level,age_band,balance_quantile,is_active,first_topup_at,first_card_payment_at,first_p2p_at,_edge_case_profile
0,u000000,2024-02-23 15:33:54+00:00,GB,premium,basic,55+,q2,True,2024-02-28 23:18:16+00:00,2024-02-29 19:13:03+00:00,2024-05-01 01:51:05+00:00,normal
1,u000001,2024-06-27 16:26:14+00:00,RO,plus,enhanced,25-34,q2,True,2024-07-31 07:09:53+00:00,2024-06-27 18:20:19+00:00,2024-07-16 07:57:54+00:00,normal
2,u000002,2024-06-18 18:29:36+00:00,RO,premium,full,18-24,q3,True,2024-07-15 02:29:31+00:00,2024-06-20 02:20:48+00:00,2024-06-25 00:32:46+00:00,normal
3,u000003,2025-08-21 14:24:40+00:00,DE,standard,full,55+,q3,True,2025-08-23 08:38:21+00:00,2025-08-25 01:25:03+00:00,2026-01-10 16:53:42+00:00,normal
4,u000004,2025-01-05 16:24:44+00:00,RO,standard,full,45-54,q3,True,2025-01-06 23:02:18+00:00,2025-01-06 07:31:21+00:00,2025-01-19 06:46:23+00:00,normal
...,...,...,...,...,...,...,...,...,...,...,...,...
495,u000495,2025-05-31 08:24:54+00:00,GB,metal,full,55+,q5,True,2025-06-02 05:09:08+00:00,2025-06-02 16:50:00+00:00,2025-07-05 09:43:20+00:00,normal
496,u000496,2024-06-10 17:12:38+00:00,RO,metal,basic,18-24,q2,True,2024-07-07 01:03:37+00:00,2024-06-13 18:47:42+00:00,2024-06-30 03:43:11+00:00,normal
497,u000497,2025-07-16 12:50:10+00:00,FR,standard,full,45-54,q5,True,2025-10-21 11:02:58+00:00,2025-08-09 22:50:10+00:00,2025-08-23 16:36:33+00:00,normal
498,u000498,2025-11-24 10:06:08+00:00,GB,premium,full,18-24,q5,True,NaT,2026-02-14 02:53:39+00:00,2026-03-16 08:17:20+00:00,normal


In [10]:
pd.read_parquet("C:/Users/levyr/Desktop/random-projects/pragma/data/raw/events.parquet")

,entity_id,event_id,created_at,type,amount,currency,direction,counterparty_id,description,view,mcc,merchant_name,channel,source_currency
0,u000000,e00000144,2024-02-28 23:18:16+00:00,topup,35.68,GBP,in,NaN,dinner with friends,NaN,NaN,NaN,open_banking,NaN
1,u000000,e00000127,2024-02-29 19:13:03+00:00,card_payment,3.44,EUR,out,NaN,metal plan,NaN,5411,Tesco,NaN,NaN
2,u000000,e00000132,2024-03-03 23:18:10+00:00,card_payment,23.45,PLN,out,NaN,dinner with friends,NaN,5812,Amazon,NaN,NaN
3,u000000,e00000197,2024-03-04 23:46:33+00:00,topup,10.70,USD,in,NaN,dinner with friends,NaN,NaN,NaN,bank_transfer,NaN
4,u000000,e00000215,2024-03-10 00:31:09+00:00,card_payment,114.91,RON,out,NaN,monthly subscription,NaN,6012,Uber,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
108017,u000499,e00107771,2026-06-29 06:02:35+00:00,card_payment,4.83,RON,out,NaN,monthly subscription,NaN,5732,Uber,NaN,NaN
108018,u000499,e00107651,2026-06-29 12:48:02+00:00,card_payment,1.84,EUR,out,NaN,rent payment,NaN,5732,Local Cafe,NaN,NaN
108019,u000499,e00108014,2026-06-29 16:15:55+00:00,card_payment,12.25,PLN,out,NaN,grocery shopping,NaN,5812,Netflix,NaN,NaN
108020,u000499,e00107907,2026-06-29 21:01:22+00:00,card_payment,3.99,EUR,out,NaN,monthly subscription,NaN,4111,Amazon,NaN,NaN
